# Klasifikasi Tulisan Tangan dengan HOG + SVM pada Dataset EMNIST

## Dataset Yang Digunakan

Dataset yang digunakan adalah **EMNIST-MNIST** (*Extended MNIST*), yaitu versi perluasan dari dataset MNIST klasik yang dikembangkan oleh Cohen et al. (2017). Dataset ini berisi kumpulan gambar tulisan tangan angka dari 0 sampai 9 yang dikumpulkan dari ribuan penulis berbeda.

File yang dipilih adalah **emnist-mnist-train.csv** yang memiliki **60.000 baris data**, di mana setiap baris mewakili satu gambar tulisan tangan. Kolom pertama adalah **label** (angka 0–9), sedangkan 784 kolom sisanya adalah **nilai pixel** dari gambar berukuran 28×28 piksel yang diratakan menjadi satu baris. Nilai pixel berkisar antara 0 hingga 255.

---

## Gambaran Program

Program ini dibuat untuk menyelesaikan tugas klasifikasi karakter tulisan tangan menggunakan dua komponen utama:

- **HOG (Histogram of Oriented Gradients)** Digunakan untuk mengekstrak fitur dari gambar. Cara Kerja HOG adalah dengan menangkap informasi arah dan kekuatan pada tepi gambar yang sangat cocok untuk mendeskripsikan bentuk karakter.
- **SVM (Support Vector Machine)** Digunakan sebagai klasifier yang mempelajari pola dari fitur HOG dan memprediksi kelas digit.

Untuk evaluasi performa model digunakan metode **Leave-One-Out Cross-Validation (LOOCV)**, di mana setiap sampel secara bergantian dijadikan data uji sementara sisanya digunakan untuk melatih model. Metrik yang ditampilkan meliputi akurasi, precision, F1-score, dan confusion matrix.

> **Catatan Tambahan:** Karena LOOCV pada 60.000 data akan membutuhkan waktu yang sangat lama, diambil subset **100 sampel** (10 per kelas) secara acak dengan seed yang tetap agar hasil dapat direproduksi.

---

## 1. Import Library

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, f1_score, classification_report
)
import warnings
warnings.filterwarnings('ignore')

c:\Users\ASUS TUF GAMING\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 2. Load Dataset

In [2]:
df = pd.read_csv('emnist-mnist-train.csv', header=None)
labels = df.iloc[:, 0].values
pixels = df.iloc[:, 1:].values.astype(np.float32)
print(f"Total samples  : {len(labels)}")
print(f"Image shape    : 28x28 (784 pixels)")
print(f"Kelas (digit)  : {np.unique(labels)}")

Total samples  : 60000
Image shape    : 28x28 (784 pixels)
Kelas (digit)  : [0 1 2 3 4 5 6 7 8 9]


## 3. Sampling Subset Data

In [3]:
np.random.seed(42)
N_PER_CLASS = 10
idx = []
for cls in np.unique(labels):
    cls_idx = np.where(labels == cls)[0]
    chosen = np.random.choice(cls_idx, N_PER_CLASS, replace=False)
    idx.extend(chosen)
idx = np.array(idx)
X_raw = pixels[idx]
y = labels[idx]
print(f"Subset ukuran  : {len(y)} sampel ({N_PER_CLASS} per kelas)")
print(f"Distribusi kelas: {dict(zip(*np.unique(y, return_counts=True)))}")

Subset ukuran  : 100 sampel (10 per kelas)
Distribusi kelas: {0: 10, 1: 10, 2: 10, 3: 10, 4: 10, 5: 10, 6: 10, 7: 10, 8: 10, 9: 10}


## 4. Visualisasi Sampel per Kelas

In [4]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.ravel()
for cls in range(10):
    sample_idx = np.where(y == cls)[0][0]
    img = X_raw[sample_idx].reshape(28, 28)
    axes[cls].imshow(img, cmap='gray')
    axes[cls].set_title(f'Digit {cls}')
    axes[cls].axis('off')
plt.suptitle('Contoh Sampel per Kelas', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=100, bbox_inches='tight')
plt.show()
print("Visualisasi sampel tersimpan.")

Visualisasi sampel tersimpan.


## 5. Ekstraksi Fitur HOG

In [5]:
HOG_PARAMS = {
    "orientations"  : 9,
    "pixels_per_cell": (4, 4),
    "cells_per_block" : (2, 2),
    "block_norm"     : "L2-Hys",
    "visualize"      : False,
}

def extract_hog(image_flat):
    img = image_flat.reshape(28, 28)
    img_norm = img / 255.0
    feat = hog(img_norm, **HOG_PARAMS)
    return feat

X_hog = np.array([extract_hog(x) for x in X_raw])
print(f"Dimensi HOG feature per sampel : {X_hog.shape[1]}")
print(f"Total dataset setelah ekstraksi : {X_hog.shape}")

Dimensi HOG feature per sampel : 1296
Total dataset setelah ekstraksi : (100, 1296)


## 6. Visualisasi HOG Feature

In [6]:
img_sample = X_raw[0].reshape(28, 28) / 255.0
feat_sample, hog_img = hog(
    img_sample,
    orientations=HOG_PARAMS["orientations"],
    pixels_per_cell=HOG_PARAMS["pixels_per_cell"],
    cells_per_block=HOG_PARAMS["cells_per_block"],
    block_norm=HOG_PARAMS["block_norm"],
    visualize=True,
)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(img_sample, cmap='gray')
ax1.set_title(f"Citra Asli - Digit {y[0]}")
ax1.axis('off')
ax2.imshow(hog_img, cmap='gray')
ax2.set_title("HOG Feature Visualization")
ax2.axis('off')
plt.tight_layout()
plt.savefig('hog_visualization.png', dpi=100, bbox_inches='tight')
plt.show()
print("Visualisasi HOG tersimpan.")

Visualisasi HOG tersimpan.


## 7. Parameter HOG dan SVM

In [7]:
SVM_PARAMS = {
    "kernel" : "rbf",
    "C"      : 10.0,
    "gamma"  : "scale",
    "decision_function_shape": "ovr",
}
print("Parameter HOG:")
for k, v in HOG_PARAMS.items():
    print(f"  {k:20s}: {v}")
print()
print("Parameter SVM:")
for k, v in SVM_PARAMS.items():
    print(f"  {k:20s}: {v}")

Parameter HOG:
  orientations        : 9
  pixels_per_cell     : (4, 4)
  cells_per_block     : (2, 2)
  block_norm          : L2-Hys
  visualize           : False

Parameter SVM:
  kernel              : rbf
  C                   : 10.0
  gamma               : scale
  decision_function_shape: ovr


## 8. Proses LOOCV

In [8]:
loo = LeaveOneOut()
y_true = []
y_pred = []

n_splits = len(y)
print(f"Memulai LOOCV dengan {n_splits} iterasi ...")

for fold, (train_idx, test_idx) in enumerate(loo.split(X_hog)):
    X_train, X_test = X_hog[train_idx], X_hog[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    clf = SVC(**SVM_PARAMS)
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    y_true.append(y_test[0])
    y_pred.append(pred[0])
    if (fold + 1) % 20 == 0:
        print(f"  Fold {fold+1}/{n_splits} selesai")

y_true = np.array(y_true)
y_pred = np.array(y_pred)
print("LOOCV selesai.")

Memulai LOOCV dengan 100 iterasi ...
  Fold 20/100 selesai
  Fold 40/100 selesai
  Fold 60/100 selesai
  Fold 80/100 selesai
  Fold 100/100 selesai
LOOCV selesai.


## 9. Hasil Evaluasi

In [9]:
accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
f1        = f1_score(y_true, y_pred, average='weighted', zero_division=0)
cm        = confusion_matrix(y_true, y_pred)

print("="*45)
print("         HASIL EVALUASI LOOCV")
print("="*45)
print(f"  Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"  Precision : {precision:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print("="*45)
print()
print("Classification Report:")
print(classification_report(y_true, y_pred, zero_division=0))

         HASIL EVALUASI LOOCV
  Accuracy  : 0.8800  (88.00%)
  Precision : 0.8897
  F1-Score  : 0.8821

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.90      0.95        10
           1       1.00      1.00      1.00        10
           2       0.64      0.70      0.67        10
           3       0.73      0.80      0.76        10
           4       0.83      1.00      0.91        10
           5       1.00      0.80      0.89        10
           6       1.00      1.00      1.00        10
           7       0.90      0.90      0.90        10
           8       1.00      0.90      0.95        10
           9       0.80      0.80      0.80        10

    accuracy                           0.88       100
   macro avg       0.89      0.88      0.88       100
weighted avg       0.89      0.88      0.88       100



## 10. Confusion Matrix

In [10]:
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
classes = [str(c) for c in np.unique(y_true)]
tick_marks = np.arange(len(classes))
ax.set_xticks(tick_marks)
ax.set_xticklabels(classes, fontsize=12)
ax.set_yticks(tick_marks)
ax.set_yticklabels(classes, fontsize=12)
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center", fontsize=12,
                color="white" if cm[i, j] > thresh else "black")
ax.set_xlabel('Prediksi', fontsize=13)
ax.set_ylabel('Aktual', fontsize=13)
ax.set_title(
    f"Confusion Matrix - LOOCV\n"
    f"Accuracy: {accuracy*100:.2f}%  |  Precision: {precision:.4f}  |  F1: {f1:.4f}",
    fontsize=13
)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print("Confusion matrix tersimpan.")

Confusion matrix tersimpan.


## 11. Akurasi Per Kelas

In [11]:
per_class_acc = []
for cls in np.unique(y_true):
    mask = y_true == cls
    acc_cls = accuracy_score(y_true[mask], y_pred[mask])
    per_class_acc.append(acc_cls)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    [f'Digit {c}' for c in np.unique(y_true)],
    [a * 100 for a in per_class_acc],
    color='steelblue', edgecolor='navy'
)
for bar, acc in zip(bars, per_class_acc):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{acc*100:.0f}%", ha='center', va='bottom', fontsize=11
    )
ax.set_xlabel('Kelas (Digit)', fontsize=13)
ax.set_ylabel('Akurasi (%)', fontsize=13)
ax.set_title('Akurasi Per Kelas - LOOCV', fontsize=14)
ax.set_ylim(0, 115)
ax.axhline(y=accuracy * 100, color='red', linestyle='--', label=f'Rata-rata: {accuracy*100:.1f}%')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=120, bbox_inches='tight')
plt.show()
print("Bar chart akurasi per kelas tersimpan.")

Bar chart akurasi per kelas tersimpan.


---

## Analisis Hasil Eksekusi

### Data & Preprocessing

Dataset EMNIST-MNIST yang dimuat memiliki 60.000 sampel dengan 784 fitur pixel per gambar. Dari jumlah itu, diambil 100 sampel secara acak masing-masing 10 sampel per kelas digit (0–9) untuk membuat LOOCV tetap komputasional. Dari visualisasi sampel, terlihat variasi tulisan tangan yang cukup tinggi antar sampel, misalnya digit 2 yang tampak seperti angka 9 terbalik dan digit 3 yang berbentuk menyerupai huruf "w". Ini yang membuat klasifikasi tulisan tangan sangat menantang.

### Ekstraksi Fitur HOG

Setiap gambar 28×28 diekstrak menggunakan HOG dengan parameter: 9 orientasi, sel berukuran 4×4 pixel, blok berukuran 2×2 sel, dan normalisasi L2-Hys. Hasilnya setiap gambar direpresentasikan sebagai vektor fitur berukuran **1.296 dimensi**, jauh lebih informatif dibanding 784 nilai pixel mentah karena HOG fokus menangkap arah gradien dan kontur bentuk karakter, bukan sekadar nilai kecerahan pixel. Dari visualisasi HOG pada digit 0, bisa terlihat pola blok-blok kecil yang merepresentasikan arah tepi dari lingkaran angka 0.

### LOOCV & SVM

Model SVM dengan kernel RBF (C=10, gamma=scale) dilatih sebanyak 100 kali satu kali per sampel. Setiap iterasi, 99 sampel dipakai untuk training dan 1 sampel sisanya untuk testing. Proses ini memastikan setiap data pernah menjadi data uji tepat satu kali.

### Hasil Evaluasi

Performa yang diperoleh cukup memuaskan:

| Metrik | Nilai |
|---|---|
| **Accuracy** | 88.00% |
| **Precision** | 0.8897 |
| **F1-Score** | 0.8821 |

Dari confusion matrix, terlihat bahwa digit **1, 4, dan 6** mendapat akurasi sempurna 100% kemungkinan karena bentuknya yang cukup unik dan mudah dibedakan. Sebaliknya, digit **2** paling sering salah diklasifikasikan sebagai digit 3, yang masuk akal melihat bentuknya yang memang bisa mirip satu sama lain tergantung gaya tulisan. Digit **9** juga sempat diprediksi sebagai 4 dan 7.

Secara keseluruhan, kombinasi HOG + SVM terbukti efektif untuk klasifikasi digit tulisan tangan, dengan performa yang solid bahkan pada subset data yang kecil.
